# Pipeline run accounting overview

This standalone notebook aggregates request, token, and provider-reported cost data for the main development runs and the final full production run. It reads the saved JSON/JSONL artifacts directly and does not import project code. Extract the pipeline release assets and set `PIPELINE_ARTIFACT_ROOT` to the resulting `qwen_iteration` directory.

**Definitions**

- **Requests** are physical API attempts, including retries and failed attempts.
- **Input tokens** are OpenRouter `prompt_tokens`; they combine text and visual-input accounting. The provider does not expose a reliable text/image split.
- **Output tokens** are OpenRouter `completion_tokens`.
- **Cost** is the sum of provider-reported usage cost. Failed attempts without usage metadata contribute zero here, so totals can be observed lower bounds rather than invoice totals.
- **Route totals** represent one runnable pipeline. **Campaign totals** combine several variants, pilots, or shared calls and should not be compared as if they were one route.


In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import json
import os

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None

def show_markdown(text):
    if Markdown is None:
        print(text)
    else:
        display(Markdown(text))

def find_artifact_root():
    configured = os.environ.get('PIPELINE_ARTIFACT_ROOT')
    if configured:
        qwen = Path(configured).expanduser().resolve()
        if qwen.is_dir():
            return qwen.parent, qwen
        raise FileNotFoundError(f'PIPELINE_ARTIFACT_ROOT is not a directory: {qwen}')
    for candidate in (Path.cwd(), *Path.cwd().parents):
        qwen = candidate / 'artifacts' / 'pipeline' / 'extracted' / 'qwen_iteration'
        if qwen.is_dir():
            return candidate, qwen
        legacy = candidate / 'qwen_iteration'
        if legacy.is_dir():
            return candidate, legacy
    raise FileNotFoundError('Extract the pipeline release assets, then set PIPELINE_ARTIFACT_ROOT to their qwen_iteration directory.')

ROOT, QWEN = find_artifact_root()
print(f'Artifact root: {QWEN}')


In [ ]:
def read_json(path):
    with Path(path).open(encoding='utf-8') as handle:
        return json.load(handle)

def iter_jsonl(path):
    with Path(path).open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f'Invalid JSON in {path}, line {line_number}') from exc

def usage_of(record):
    return record.get('usage') or (record.get('response_metadata') or {}).get('usage') or {}

def relative(path):
    return str(Path(path).resolve().relative_to(ROOT.resolve())).replace('\\', '/')

def finish_row(row):
    images = row.get('images') or 0
    requests = row.get('requests') or 0
    successful = row.get('successful_requests') or 0
    row['failed_requests'] = requests - successful
    row['success_rate'] = successful / requests if requests else None
    row['calls_per_image'] = requests / images if images else None
    row['input_tokens_per_image'] = row['input_tokens'] / images if images else None
    row['output_tokens_per_image'] = row['output_tokens'] / images if images else None
    row['cost_per_image_usd'] = row['cost_usd'] / images if images else None
    return row

def summarize_ledger(label, path, images, accounting_level, include=None, note=''):
    totals = defaultdict(float)
    for record in iter_jsonl(path):
        if include is not None and not include(record):
            continue
        totals['requests'] += 1
        totals['successful_requests'] += int(record.get('ok') is True)
        usage = usage_of(record)
        totals['input_tokens'] += usage.get('prompt_tokens') or 0
        totals['output_tokens'] += usage.get('completion_tokens') or 0
        totals['total_tokens'] += usage.get('total_tokens') or 0
        totals['cost_usd'] += usage.get('cost', usage.get('cost_usd', 0)) or 0
    return finish_row({
        'run': label, 'accounting_level': accounting_level, 'images': images,
        'complete_images': None,
        'requests': int(totals['requests']),
        'successful_requests': int(totals['successful_requests']),
        'input_tokens': int(totals['input_tokens']),
        'output_tokens': int(totals['output_tokens']),
        'total_tokens': int(totals['total_tokens']),
        'cost_usd': float(totals['cost_usd']),
        'support': 'raw request ledger', 'source': relative(path), 'note': note,
    })

def markdown_table(rows, columns, labels=None):
    labels = labels or {column: column for column in columns}
    def clean(value):
        return str(value).replace('|', '&#124;').replace('\n', ' ')
    lines = [
        '| ' + ' | '.join(clean(labels.get(column, column)) for column in columns) + ' |',
        '| ' + ' | '.join('---' for _ in columns) + ' |',
    ]
    for row in rows:
        lines.append('| ' + ' | '.join(clean(row.get(column, '—')) for column in columns) + ' |')
    return '\n'.join(lines)

def format_summary(rows):
    formatted = []
    for row in rows:
        formatted.append({
            'run': row['run'],
            'level': row['accounting_level'],
            'images': f"{row['images']:,}",
            'requests': f"{row['requests']:,}",
            'failed': f"{row['failed_requests']:,}",
            'calls/image': f"{row['calls_per_image']:.3f}",
            'input tokens': f"{row['input_tokens']:,}",
            'output tokens': f"{row['output_tokens']:,}",
            'total tokens': f"{row['total_tokens']:,}",
            'cost (USD)': f"${row['cost_usd']:,.6f}",
            'cost/image': f"${row['cost_per_image_usd']:.6f}",
        })
    return formatted


## Main development and production accounting

The first-100 P1–P4 rows come from the generated cost-analysis JSON, which was itself computed from the request ledger. The other rows are summed directly from raw ledgers, except completeness of the final production run, which comes from its status snapshot.


In [ ]:
rows = []

rows.append(summarize_ledger(
    'Early single-shot (first 20)',
    QWEN / 'output/first20_single_shot.jsonl', 20, 'route attempt',
    note='Early page-level baseline; five records are marked unsuccessful.'
))

cost_path = QWEN / 'first100_pipeline_matrix/evaluation/first100_frozen_v1/cost_analysis/cost_analysis.json'
cost_data = read_json(cost_path)
for pipeline_name, values in cost_data['pipelines'].items():
    rows.append(finish_row({
        'run': f"First-100 {pipeline_name.upper()}",
        'accounting_level': 'route total',
        'images': int(cost_data['pages']), 'complete_images': None,
        'requests': int(values['attempts']), 'successful_requests': int(values['ok']),
        'input_tokens': int(values['prompt_tokens']),
        'output_tokens': int(values['completion_tokens']),
        'total_tokens': int(values['total_tokens']),
        'cost_usd': float(values['reported_cost_usd']),
        'support': 'generated structured cost analysis', 'source': relative(cost_path),
        'note': cost_data['accounting_note'],
    }))

rows.append(summarize_ledger(
    'Next-round 200 (all experimental calls)',
    QWEN / 'next_round_200/output/shared_request_ledger.jsonl', 200, 'campaign total',
    note='Shared inference, promoted routes, and rejected pilots; not the cost of one route.'
))
rows.append(summarize_ledger(
    'Lean-final 280 (all tested variants)',
    QWEN / 'lean_final_280/output/request_ledger.jsonl', 280, 'campaign total',
    note='Includes S1/S2/S2n variants and retry overhead; not the selected-route expectation.'
))
rows.append(summarize_ledger(
    'Final-refinement 280 (selected ablations)',
    QWEN / 'final_refinement_280/output/request_ledger.jsonl', 280, 'campaign total',
    note='Tests targeted subsets and several stages; calls/image is campaign spend divided by 280 candidate pages.'
))
rows.append(summarize_ledger(
    'Frozen final candidate (398 pages, main route)',
    QWEN / 'final_pipeline_standalone/output/request_ledger.jsonl', 398, 'route total',
    include=lambda record: record.get('stage') in {'structure', 'entities'},
    note='Excludes the optional duplicate sidecar recorded in the same ledger.'
))
rows.append(summarize_ledger(
    'No-tiles ablation (398 pages)',
    QWEN / 'final_pipeline_standalone/output/experiments/no_tiles_s1/request_ledger.jsonl',
    398, 'route total', note='Comparable structural ablation of the frozen route.'
))

production_root = QWEN / 'final_pipeline_standalone/output/production/full_pages_joined_v1'
production_status_path = production_root / 'run_status.json'
production_ledger_path = production_root / 'request_ledger.jsonl'
production_status = read_json(production_status_path)
production_pages = int(production_status['cumulative']['manifest_pages'])
production_row = summarize_ledger(
    'Final full production run', production_ledger_path, production_pages, 'route total',
    note='Request/token/cost totals use the newest raw ledger; page completeness uses run_status.json.'
)
production_row['complete_images'] = int(production_status['cumulative']['complete_assembled_pages'])
production_row['support'] = 'raw request ledger + structured status snapshot'
production_row['source'] = relative(production_ledger_path) + ' + ' + relative(production_status_path)
rows.append(production_row)

summary_columns = ['run', 'level', 'images', 'requests', 'failed', 'calls/image', 'input tokens', 'output tokens', 'total tokens', 'cost (USD)', 'cost/image']
show_markdown(markdown_table(format_summary(rows), summary_columns))


## Final-run completeness and status/ledger consistency

The production ledger was appended after the most recent status snapshot. The cell below reports this explicitly, rather than silently mixing the two timestamps.


In [ ]:
complete = production_row['complete_images']
pages = production_row['images']
coverage = complete / pages
status_attempts = int(production_status['request_ledger']['physical_attempts_all_invocations'])
ledger_attempts = production_row['requests']
unrecovered = int(production_status['unrecovered']['structure_count'])
retryable = int(production_status['terminal_unrecovered']['retryable_task_count'])
show_markdown(
    f"- Fully assembled pages in the status snapshot: **{complete:,} / {pages:,} ({coverage:.3%})**.\n"
    f"- `all_selected_complete`: **{production_status['all_selected_complete']}**.\n"
    f"- Unrecovered structure pages: **{unrecovered}**; marked retryable: **{retryable}**.\n"
    f"- Physical attempts in status snapshot: **{status_attempts:,}**. Newest raw ledger: **{ledger_attempts:,}** "
    f"(**{ledger_attempts - status_attempts:+,}** later attempts).\n"
    f"- Therefore this is a **near-complete final full run**, not evidence of literal 100% completion."
)


## Final pipeline stage breakdown

This table shows why calls per page exceed one: the final pipeline uses a page-level structure stage followed by entity-level calls, with retries counted as additional physical requests. The frozen evaluation ledger also contains an optional duplicate sidecar, shown separately. `Unique task keys` can be lower than the page count when the same image identifier occurs in both evaluation cohorts; this happens once in the frozen 398-page set.


In [ ]:
def summarize_stages(run_label, path, images):
    groups = defaultdict(lambda: {
        'requests': 0, 'successful': 0, 'input_tokens': 0,
        'output_tokens': 0, 'total_tokens': 0, 'cost_usd': 0.0, 'task_keys': set(),
    })
    for record in iter_jsonl(path):
        stage = record.get('stage') or 'unspecified'
        group = groups[stage]
        group['requests'] += 1
        group['successful'] += int(record.get('ok') is True)
        logical_key = record.get('task_key') or record.get('logical_key') or record.get('case_id') or record.get('image_id')
        if logical_key is not None:
            group['task_keys'].add(str(logical_key))
        usage = usage_of(record)
        group['input_tokens'] += usage.get('prompt_tokens') or 0
        group['output_tokens'] += usage.get('completion_tokens') or 0
        group['total_tokens'] += usage.get('total_tokens') or 0
        group['cost_usd'] += usage.get('cost', usage.get('cost_usd', 0)) or 0
    result = []
    for stage, group in sorted(groups.items()):
        result.append({
            'run': run_label, 'stage': stage, 'unique task keys': len(group['task_keys']),
            'requests': group['requests'], 'successful': group['successful'],
            'failed': group['requests'] - group['successful'],
            'calls/image': group['requests'] / images,
            'input tokens': group['input_tokens'], 'output tokens': group['output_tokens'],
            'cost (USD)': group['cost_usd'],
        })
    return result

stage_rows = summarize_stages(
    'Frozen 398', QWEN / 'final_pipeline_standalone/output/request_ledger.jsonl', 398
) + summarize_stages('Production', production_ledger_path, production_pages)

formatted_stages = []
for row in stage_rows:
    formatted_stages.append({
        **row,
        'unique task keys': f"{row['unique task keys']:,}",
        'requests': f"{row['requests']:,}", 'successful': f"{row['successful']:,}",
        'failed': f"{row['failed']:,}", 'calls/image': f"{row['calls/image']:.3f}",
        'input tokens': f"{row['input tokens']:,}", 'output tokens': f"{row['output tokens']:,}",
        'cost (USD)': f"${row['cost (USD)']:,.6f}",
    })
stage_columns = ['run', 'stage', 'unique task keys', 'requests', 'successful', 'failed', 'calls/image', 'input tokens', 'output tokens', 'cost (USD)']
show_markdown(markdown_table(formatted_stages, stage_columns))


## Evidence and comparability notes

Every numerical row below names its supporting artifact. This makes it possible to distinguish raw-log claims from generated reports and to avoid treating experimental campaign spend as route performance.


In [ ]:
evidence_rows = [{
    'run': row['run'], 'support': row['support'], 'source': row['source'], 'interpretation': row['note']
} for row in rows]
show_markdown(markdown_table(evidence_rows, ['run', 'support', 'source', 'interpretation']))


## Ledger-backed recorded minimum across all modern iteration runs

This is the defensible cumulative record. It counts every retained `request_ledger.jsonl`, `shared_request_ledger.jsonl`, and `call_ledger.jsonl` inside `qwen_iteration/` exactly once. Derived result files are excluded because they repeat calls already represented in the ledgers. Legacy experiments without a central ledger are not included, so this remains a recorded minimum rather than a claim about complete historical spend.


In [ ]:
def sum_ledger_files(paths):
    totals = defaultdict(float)
    for path in paths:
        totals['ledger_files'] += 1
        for record in iter_jsonl(path):
            totals['requests'] += 1
            totals['successful'] += int(record.get('ok') is True)
            usage = usage_of(record)
            totals['input_tokens'] += usage.get('prompt_tokens') or 0
            totals['output_tokens'] += usage.get('completion_tokens') or 0
            totals['total_tokens'] += usage.get('total_tokens') or 0
            totals['cost_usd'] += usage.get('cost', usage.get('cost_usd', 0)) or 0
    return totals

all_ledger_paths = sorted(
    set(QWEN.rglob('request_ledger.jsonl'))
    | set(QWEN.rglob('shared_request_ledger.jsonl'))
    | set(QWEN.rglob('call_ledger.jsonl'))
)
production_ledger_paths = [production_ledger_path]
development_ledger_paths = [path for path in all_ledger_paths if path not in production_ledger_paths]

all_recorded = sum_ledger_files(all_ledger_paths)
production_recorded = sum_ledger_files(production_ledger_paths)
development_recorded = sum_ledger_files(development_ledger_paths)

def cumulative_row(label, values):
    requests = int(values['requests'])
    successful = int(values['successful'])
    return {
        'scope': label, 'ledger files': f"{int(values['ledger_files']):,}",
        'requests': f"{requests:,}", 'successful': f"{successful:,}",
        'failed': f"{requests - successful:,}",
        'input tokens': f"{int(values['input_tokens']):,}",
        'output tokens': f"{int(values['output_tokens']):,}",
        'total tokens': f"{int(values['total_tokens']):,}",
        'cost (USD)': f"${values['cost_usd']:,.6f}",
    }

cumulative_rows = [
    cumulative_row('Development ledgers', development_recorded),
    cumulative_row('Final production ledger', production_recorded),
    cumulative_row('All modern iteration ledgers — recorded minimum', all_recorded),
]
cumulative_columns = ['scope', 'ledger files', 'requests', 'successful', 'failed', 'input tokens', 'output tokens', 'total tokens', 'cost (USD)']
show_markdown(markdown_table(cumulative_rows, cumulative_columns))


## Optional CSV export

Set `EXPORT_CSV = True` to write the main summary beside this notebook. The notebook itself remains read-only by default.


In [ ]:
EXPORT_CSV = False
if EXPORT_CSV:
    export_path = Path('pipeline_run_accounting_overview.csv').resolve()
    fieldnames = list(rows[0].keys())
    with export_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Wrote {export_path}')
else:
    print('CSV export disabled.')
